# 01 - Data Loading & Cleaning
**Objective:** Load raw datasets from World Bank and WTO, clean inconsistencies, handle missing values, and save standardized outputs.
**Data Sources:**
- World Bank Open Data – GDP, GDP growth, GDP per capita, population, trade (% of GDP), imports (% of GDP)
- WTO Statistics – Merchandise trade values by product group for Algeria
**Output:** Cleaned CSV files written to `data/processed/`.

In [ ]:
import os, sys, numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
PROJECT_ROOT = Path.cwd().parent.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Raw data dir:', RAW_DIR)
print('Processed dir:', PROCESSED_DIR)

## 1.2 – Load & Clean World Bank Data
Indicators (wide format with years as columns):
- GDP (current US$) – `NY.GDP.MKTP.CD`
- GDP growth (annual %) – `NY.GDP.MKTP.KD.ZG`
- GDP per capita (current US$) – `NY.GDP.PCAP.CD`
- Population, total – `SP.POP.TOTL`
- Merchandise trade (% of GDP) – `TG.VAL.TOTL.GD.ZS`
- Imports of goods and services (% of GDP) – `NE.IMP.GNFS.ZS`

In [ ]:
wb_indicators = {
    'gdp-current-usd': ('GDP (current US$)', 'NY.GDP.MKTP.CD'),
    'gdp-growth-rate': ('GDP growth (annual %)', 'NY.GDP.MKTP.KD.ZG'),
    'gdp-per-capita': ('GDP per capita (current US$)', 'NY.GDP.PCAP.CD'),
    'population': ('Population, total', 'SP.POP.TOTL'),
    'trade-percentage-of-gdp': ('Merchandise trade (% of GDP)', 'TG.VAL.TOTL.GD.ZS'),
    'import-of-gdp': ('Imports of goods and services (% of GDP)', 'NE.IMP.GNFS.ZS'),
}
wb_base = RAW_DIR / 'worldbank'
def load_worldbank_indicator(folder, indicator_name, indicator_code):
    folder_path = wb_base / folder
    candidates = list(folder_path.glob('API_*.csv'))
    if not candidates:
        raise FileNotFoundError(f'No API CSV found in {folder_path}')
    filepath = candidates[0]
    df = pd.read_csv(filepath, skiprows=4, low_memory=False)
    df.columns = df.columns.str.strip()
    if 'Indicator Name' in df.columns:
        df = df[df['Indicator Name'] == indicator_name]
    id_vars = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
    id_vars = [c for c in id_vars if c in df.columns]
    year_cols = [c for c in df.columns if c not in id_vars]
    df_long = df.melt(id_vars=id_vars, value_vars=year_cols, var_name='year', value_name='value')
    df_long['year'] = pd.to_numeric(df_long['year'], errors='coerce')
    df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')
    df_long = df_long.dropna(subset=['year', 'value']).reset_index(drop=True)
    df_long.rename(columns={'Country Name': 'country', 'Country Code': 'country_code'}, inplace=True)
    df_long['indicator'] = indicator_code
    df_long['indicator_name'] = indicator_name
    return df_long[['country', 'country_code', 'year', 'indicator', 'indicator_name', 'value']]
wb_frames = []
for folder, (ind_name, ind_code) in wb_indicators.items():
    try:
        df_wb = load_worldbank_indicator(folder, ind_name, ind_code)
        wb_frames.append(df_wb)
        print(f'Loaded {folder}: {df_wb.shape[0]} rows')
    except Exception as e:
        print(f'ERROR loading {folder}: {e}')
df_wb_long = pd.concat(wb_frames, ignore_index=True) if wb_frames else pd.DataFrame()
print('\nCombined World Bank long-format shape:', df_wb_long.shape)
display(df_wb_long.head(10))

In [ ]:
exclude_keywords = ['income', 'only', 'dividend', 'union', 'world', 'africa', 'asia', 'europe', 'america', 'arab', 'caribbean', 'pacific', 'small states', 'fragile', 'oecd', 'euro area', 'european union']
mask = ~df_wb_long['country'].str.lower().str.contains('|'.join(exclude_keywords), na=False)
df_wb_long_clean = df_wb_long[mask].copy().reset_index(drop=True)
df_wb_long_clean = df_wb_long_clean[df_wb_long_clean['year'] >= 2000].reset_index(drop=True)
print('After filtering aggregate regions & years>=2000:', df_wb_long_clean.shape)
print('Year range:', int(df_wb_long_clean['year'].min()), '-', int(df_wb_long_clean['year'].max()))
print('Unique countries:', df_wb_long_clean['country'].nunique())
print('Unique indicators:', df_wb_long_clean['indicator'].unique())
display(df_wb_long_clean.head())

In [ ]:
df_wb_wide = df_wb_long_clean.pivot_table(
    index=['country', 'country_code', 'year'],
    columns='indicator',
    values='value',
    aggfunc='first'
).reset_index()
df_wb_wide.columns.name = None
df_wb_wide.columns = [str(c) for c in df_wb_wide.columns]
print('World Bank wide shape:', df_wb_wide.shape)
display(df_wb_wide.head())
wb_out = PROCESSED_DIR / '01_worldbank_cleaned.csv'
df_wb_wide.to_csv(wb_out, index=False)
print('Saved:', wb_out)

## 1.3 – Load & Clean WTO Data
File: `WtoData_20260325191317.csv`
Content: Merchandise trade values by product group (SITC Rev. 3 aggregates) for Algeria.

In [ ]:
wto_path = RAW_DIR / 'wto_algeria' / 'WtoData_20260325191317.csv'
df_wto = pd.read_csv(wto_path, encoding='utf-8', encoding_errors='replace', low_memory=False)
df_wto.columns = df_wto.columns.str.strip().str.lower().str.replace(' ', '_')
print('WTO shape:', df_wto.shape)
print('Columns:', df_wto.columns.tolist())
display(df_wto.head())

In [ ]:
def clean_wto(df):
    useful = [
        'reporting_economy', 'reporting_economy_iso3a_code',
        'partner_economy', 'product/sector', 'product/sector_code',
        'indicator', 'year', 'value', 'unit'
    ]
    available = [c for c in useful if c in df.columns]
    df = df[available].copy()
    rename_map = {
        'reporting_economy': 'country',
        'reporting_economy_iso3a_code': 'country_code',
        'partner_economy': 'partner',
        'product/sector': 'product_sector',
        'product/sector_code': 'product_sector_code',
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip().str.title()
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df = df.dropna(subset=['value']).reset_index(drop=True)
    df['source'] = 'wto'
    return df
df_wto_clean = clean_wto(df_wto)
print('Cleaned WTO shape:', df_wto_clean.shape)
print('Year range:', int(df_wto_clean['year'].min()), '-', int(df_wto_clean['year'].max()))
print('Unique product sectors:', df_wto_clean['product_sector'].nunique() if 'product_sector' in df_wto_clean.columns else 'N/A')
display(df_wto_clean.head(10))

In [ ]:
if 'indicator' in df_wto_clean.columns:
    print(df_wto_clean['indicator'].value_counts())
wto_out = PROCESSED_DIR / '01_wto_cleaned.csv'
df_wto_clean.to_csv(wto_out, index=False)
print('Saved:', wto_out)

## 1.4 – Missing-Value Audit

In [ ]:
def missing_summary(df, name):
    print(f'\n=== {name} ===')
    missing = df.isnull().mean().sort_values(ascending=False) * 100
    missing = missing[missing > 0]
    if missing.empty:
        print('No missing values.')
    else:
        print(missing.round(2).to_string())
missing_summary(df_wb_wide, 'World Bank')
missing_summary(df_wto_clean, 'WTO')

## 1.5 – Next Steps
Cleaned datasets persisted in `data/processed/`:
- `01_worldbank_cleaned.csv`
- `01_wto_cleaned.csv`
Proceed to notebook `02_feature_engineering.ipynb` for metric construction.